In [8]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd

from src.data import load_all
from src.model import train_model, predict
from src.dare import dare_batch
from src.metrics import evaluate_dare

np.random.seed(0)

In [9]:
splits = load_all()
for name, (X, y) in splits.items():
    print(f"{name}: X {X.shape}, y range [{y.min():.1f}, {y.max():.1f}]")

dTrain: X (250250, 2), y range [606.8, 719.7]
dTest1: X (340340, 2), y range [606.8, 718.7]
dTest2: X (340340, 2), y range [606.6, 723.2]
dTest3: X (340340, 2), y range [606.7, 721.5]


In [6]:
X_train, y_train = splits["dTrain"]
model, scalers = train_model(X_train, y_train, epochs=200)

y_hat_train = predict(model, X_train, scalers)
train_rmse = np.sqrt(np.mean((y_hat_train - y_train) ** 2))
print(f"Train RMSE: {train_rmse:.2f} °C")

  epoch  50/200  loss=0.00043
  epoch 100/200  loss=0.00040
  epoch 150/200  loss=0.00038
  epoch 200/200  loss=0.00037
Train RMSE: 0.52 °C


In [7]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from src.dare import dare_batch, dare_batch_fast

np.random.seed(0)
X_train_t = np.random.randn(50, 2)
y_train_t = np.random.randn(50)
X_test_t = np.random.randn(20, 2)
y_pred_t = np.random.randn(20)
x_d_t = np.array([1.0, 1.0, 1.0])

p1, a1 = dare_batch(X_test_t, y_pred_t, X_train_t, y_train_t, x_d_t, N=3)
p2, a2 = dare_batch_fast(X_test_t, y_pred_t, X_train_t, y_train_t, x_d_t, N=3)

print("Max PADoC diff:", np.max(np.abs(p1 - p2)))
print("Decisions match:", np.array_equal(a1, a2))

Max PADoC diff: 5.551115123125783e-17
Decisions match: True


In [10]:
from src.dare import dare_batch_fast

x_d = np.array([20.0, 20.0, 20.0])
tolerance = 10.0

results = {}
for name in ["dTest1", "dTest2", "dTest3"]:
    print(f"Running DARE on {name}...")
    X_test, y_test = splits[name]
    y_pred = predict(model, X_test, scalers)

    padocs, accepted = dare_batch_fast(
        X_test=X_test, y_pred=y_pred,
        X_train=X_train, y_train=y_train,
        x_d=x_d, N=3, Q_X=1.0, xi=0.5,
    )
    metrics = evaluate_dare(y_test, y_pred, accepted, tolerance)
    results[name] = metrics
    print(f"  {name}: f_degrade={metrics['f_degrade']:.3f}  f_peril={metrics['f_peril']:.3f}  F1={metrics['F1']:.3f}  F_max={metrics['F_max']:.3f}")

Running DARE on dTest1...
  dTest1: f_degrade=0.000  f_peril=0.000  F1=1.000  F_max=1.000
Running DARE on dTest2...
  dTest2: f_degrade=0.001  f_peril=0.000  F1=1.000  F_max=1.000
Running DARE on dTest3...
  dTest3: f_degrade=0.000  f_peril=0.000  F1=1.000  F_max=1.000
